# Dynamic SX Model — Nd/Pr Separation (PC88A/HCl)

**Reference:** Lyon, T. D.; Utgikar, V. P.; Greenhalgh, M. R. (2017). *Modeling Solvent Extraction Processes for Rare Earth Separations.* Ind. Eng. Chem. Res., 56(14), 3987–3996.

---

## Circuit Topology (Lyon et al. Figure 3 — 10 stages)

```
ORGANIC FLOW (countercurrent, left → right, recycles stage 10 → stage 1)
──────────────────────────────────────────────────────────────────────────────────────────────────────
┌──────────┐  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌──────────┐
│  Stage 1 │←─│  Stage 2 │←─│  Stage 3 │←─│  Stage 4 │←─│  Stage 5 │←─│  Stage 6 │←─│  Stage 7 │←─│  Stage 8 │←─│  Stage 9 │←─│ Stage 10 │
└──────────┘  └──────────┘  └──────────┘  └──────────┘  └──────────┘  └──────────┘  └──────────┘  └──────────┘  └──────────┘  └──────────┘
     │↑              │             │             │             │             │↑            │↓            │             │             │↑
Raffinate       Aq Feed       ←Aq Scrub→    ←Aq Scrub→    ←Aq Scrub→    Scrub Feed   Strip Pdt    ←Aq Strip→    ←Aq Strip→    Strip Feed
 (exits)     Nd=7.88 g/L                                               Nd=9.49 g/L    (exits)                                 H=1.50 mol/L
             Pr=2.07 g/L                                               H=0.10 mol/L
             H=0.01 mol/L

├─────────────────────┤├─────────────────────────────────────────────┤├───────────────────────────────────────────────┤
  EXTRACTION (1–2)                  SCRUB (3–6)                                   STRIP (7–10)
```

**Aqueous flows** are countercurrent to organic (right→left in Extraction, left→right in Scrub/Strip sections as defined by feed/exit points).


## Theory

### Extraction Equilibrium

PC88A acts as a dimer $(\overline{\text{HA}})_2$ in the organic phase. The distribution coefficient for species $i$ (Nd or Pr) is:

$$D_i = \frac{K_{\text{eq},i}([\text{H}^+])\cdot [\overline{\text{HA}}_2]_{\text{free}}^3}{[\text{H}^+]^3}$$

where the equilibrium constant has an empirical acid dependence:

$$K_{\text{eq},i}([\text{H}^+]) = a_i \cdot [\text{H}^+]^{b_i}$$

Combined:

$$D_i = a_i \cdot [\text{H}^+]^{b_i} \cdot L_{\text{free}}^3 \cdot [\text{H}^+]^{-3}$$

**Free ligand** (dimer basis, mol/L):

$$L_{\text{free}} = \max\!\left(L_{\text{total}} - 3\left(\frac{\bar{C}_{\text{Nd}}}{M_{\text{Nd}}} + \frac{\bar{C}_{\text{Pr}}}{M_{\text{Pr}}}\right),\ L_{\text{floor}}\right)$$

### Per-Stage Mass Balance

At each stage $s$, given aqueous inlet $C_{\text{in},i}$ and organic inlet $\bar{C}_{\text{in},i}$, with volumetric flow rates $v_O$ (organic) and $v_{\text{aq}}$ (aqueous, section-dependent):

$$C_{\text{ss},i} = \frac{v_O\,\bar{C}_{\text{in},i} + v_{\text{aq}}\,C_{\text{in},i}}{v_O\,D_i + v_{\text{aq}}}$$

$$\bar{C}_{\text{target},i} = D_i\,C_{\text{ss},i}$$

**Acid balance** (3 mol H⁺ released per mol REE loaded into organic):

$$[\text{H}^+]_{\text{ss}} = \max\!\left([\text{H}^+]_{\text{in}} + \frac{3\,v_O}{v_{\text{aq}}}\sum_i\frac{\bar{C}_{\text{target},i} - \bar{C}_{\text{in},i}}{M_i},\ [\text{H}^+]_{\text{floor}}\right)$$

### Numerical Method — Relaxed Iteration

The solver uses **separate-target relaxation** at each time step (row). For each stage:

1. Compute $D_i$ using the **previous-row** $[\text{H}^+]$ and $\bar{C}_i$.
2. Compute $C_{\text{ss},i}$, $\bar{C}_{\text{target},i}$, $[\text{H}^+]_{\text{ss}}$ — all **unrelaxed** targets.
3. Apply relaxation $\alpha$ to each variable **separately** toward its own target:

$$x_{\text{new}} = x_{\text{old}} + \alpha\,(x_{\text{target}} - x_{\text{old}})$$

> **Why not chain relaxation?** $D_i$ scales as $[\text{H}^+]^{b_i-3}$ (effective exponent $-4$ to $-5$). Near $[\text{H}^+]\approx 0$, using an already-relaxed $C_{\text{ss}}$ to recompute $\bar{C} = D\cdot C_{\text{new}}$ introduces a second relaxation step that catastrophically amplifies errors. The separate-target pattern is unconditionally stable from a cold start.

### Perturbation System

Feed variables and flow rates can be perturbed during a time window $[t_{\text{ini}}, t_{\text{fin}}]$ via multiplicative factors:

$$v_{\text{actual}} = v_{\text{nominal}} \times \left(1 + (m - 1)\cdot\mathbf{1}_{[t_{\text{ini}},t_{\text{fin}}]}(t)\right)$$

where $m$ is the multiplier (default 1.0 = no change). All 10 multipliers (feed concentrations + flow rates) are independently configurable.

In [ ]:
!pip install plotly ipywidgets --quiet

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML
from typing import Dict, Any

## Parameters

All model parameters are defined below as named Python variables, organized by category. These values are replicated in the interactive widget panel above and serve as the authoritative defaults.

In [ ]:
# --- Equilibrium (Keq = a * [H+]^b, Lyon et al. Fig. 2 fit) ---
a_Nd         = 0.0029
b_Nd         = -1.11
a_Pr         = 0.0004
b_Pr         = -1.83
LIGAND_TOTAL = 0.5       # mol/L dimer basis (1 M monomer, fully dimerized)
LIG_FLOOR    = 1e-6      # mol/L
MW_Nd        = 144.24    # g/mol
MW_Pr        = 140.91    # g/mol

# --- Flow Rates (L/min) ---
vO           = 0.020     # organic, constant
vAq_ext      = 0.020     # aqueous, stages 1-2 (extraction)
vAq_scrub    = 0.010     # aqueous, stages 3-6 (scrub)
vAq_strip    = 0.004     # aqueous, stages 7-10 (strip)

# --- Holdup Volumes (L per stage) ---
V_O          = 0.3
V_Aq         = 0.3

# --- External Feeds (concentrations) ---
# Aqueous feed (stage 2)
Nd_aq_feed   = 7.88     # g/L
Pr_aq_feed   = 2.07     # g/L
H_aq_feed    = 0.01     # mol/L

# Scrub feed (stage 6)
Nd_scrub_feed = 9.49    # g/L
Pr_scrub_feed = 0.0     # g/L
H_scrub_feed  = 0.10    # mol/L

# Strip feed (stage 10)
Nd_strip_feed = 0.0     # g/L
Pr_strip_feed = 0.0     # g/L
H_strip_feed  = 1.50    # mol/L

# --- Solver ---
relax        = 0.25     # relaxation factor
n_iter       = 400      # number of time steps
dt           = 0.25     # h per step  →  100 h total
C_init       = 0.0      # g/L, cold start
H_init       = 0.001    # mol/L, avoids H=0 singularity
H_FLOOR      = 1e-4     # mol/L
tol          = 1e-8     # convergence criterion (max |delta C|)

# --- Perturbation (default: +30% Nd feed for 20 h, after steady state) ---
t_ini        = 40.0     # h
t_fin        = 60.0     # h
# Multipliers (1.0 = no change)
m_Nd_aq      = 1.3
m_Pr_aq      = 1.0
m_H_aq       = 1.0
m_Nd_scrub   = 1.0
m_H_scrub    = 1.0
m_H_strip    = 1.0
m_vO         = 1.0
m_vAq_ext    = 1.0
m_vAq_scrub  = 1.0
m_vAq_strip  = 1.0

## Interactive Parameter Editor

Use the **FloatText** boxes below to enter exact numerical values (like in a spreadsheet). When you are ready, click **RUN MODEL** to execute the simulation with the current parameters. The plots and tables will appear below.

In [ ]:
def _ft(value, description, width='160px', fmt='.6g'):
    """Helper: FloatText widget with fixed layout."""
    return widgets.FloatText(
        value=value,
        description=description,
        layout=widgets.Layout(width='320px'),
        style={'description_width': '150px'},
    )

def _it(value, description):
    """Helper: IntText widget."""
    return widgets.IntText(
        value=value,
        description=description,
        layout=widgets.Layout(width='320px'),
        style={'description_width': '150px'},
    )

def _header(text):
    return widgets.HTML(f'<b style="font-size:13px;color:#1F497D">{text}</b>')

# ── Group 1: Equilibrium ──────────────────────────────────────────────────────
w_a_Nd         = _ft(a_Nd,          'a_Nd')
w_b_Nd         = _ft(b_Nd,          'b_Nd')
w_a_Pr         = _ft(a_Pr,          'a_Pr')
w_b_Pr         = _ft(b_Pr,          'b_Pr')
w_LIGAND_TOTAL = _ft(LIGAND_TOTAL,  'Ligand total (mol/L)')

grp1 = widgets.VBox([
    _header('Equilibrium  —  Keq = a · [H⁺]^b'),
    widgets.HBox([w_a_Nd, w_b_Nd]),
    widgets.HBox([w_a_Pr, w_b_Pr]),
    w_LIGAND_TOTAL,
])

# ── Group 2: Flow Rates ───────────────────────────────────────────────────────
w_vO         = _ft(vO,        'vO (L/min)')
w_vAq_ext    = _ft(vAq_ext,   'vAq ext (L/min)')
w_vAq_scrub  = _ft(vAq_scrub, 'vAq scrub (L/min)')
w_vAq_strip  = _ft(vAq_strip, 'vAq strip (L/min)')

grp2 = widgets.VBox([
    _header('Flow Rates (L/min)'),
    widgets.HBox([w_vO, w_vAq_ext]),
    widgets.HBox([w_vAq_scrub, w_vAq_strip]),
])

# ── Group 3: External Feeds ───────────────────────────────────────────────────
w_Nd_aq    = _ft(Nd_aq_feed,   'Nd aq feed (g/L)')
w_Pr_aq    = _ft(Pr_aq_feed,   'Pr aq feed (g/L)')
w_H_aq     = _ft(H_aq_feed,    'H aq feed (mol/L)')
w_Nd_scrub = _ft(Nd_scrub_feed,'Nd scrub (g/L)')
w_H_scrub  = _ft(H_scrub_feed, 'H scrub (mol/L)')
w_H_strip  = _ft(H_strip_feed, 'H strip (mol/L)')

grp3 = widgets.VBox([
    _header('External Feeds'),
    widgets.HTML('<i style="font-size:11px">Aqueous feed (stage 2)</i>'),
    widgets.HBox([w_Nd_aq, w_Pr_aq, w_H_aq]),
    widgets.HTML('<i style="font-size:11px">Scrub feed (stage 6)</i>'),
    widgets.HBox([w_Nd_scrub, w_H_scrub]),
    widgets.HTML('<i style="font-size:11px">Strip feed (stage 10)</i>'),
    w_H_strip,
])

# ── Group 4: Solver ───────────────────────────────────────────────────────────
w_relax  = _ft(relax,  'Relaxation factor')
w_n_iter = _it(n_iter, 'Iterations')
w_dt     = _ft(dt,     'dt (h/step)')
w_H_init = _ft(H_init, 'H init (mol/L)')
w_tol    = _ft(tol,    'Tolerance')

grp4 = widgets.VBox([
    _header('Solver'),
    widgets.HBox([w_relax, w_n_iter]),
    widgets.HBox([w_dt, w_H_init]),
    w_tol,
])

# ── Group 5: Perturbation ─────────────────────────────────────────────────────
w_t_ini       = _ft(t_ini,      't_ini (h)')
w_t_fin       = _ft(t_fin,      't_fin (h)')
w_m_Nd_aq     = _ft(m_Nd_aq,    'm_Nd_aq')
w_m_Pr_aq     = _ft(m_Pr_aq,    'm_Pr_aq')
w_m_H_aq      = _ft(m_H_aq,     'm_H_aq')
w_m_Nd_scrub  = _ft(m_Nd_scrub, 'm_Nd_scrub')
w_m_H_scrub   = _ft(m_H_scrub,  'm_H_scrub')
w_m_H_strip   = _ft(m_H_strip,  'm_H_strip')
w_m_vO        = _ft(m_vO,       'm_vO')
w_m_vAq_ext   = _ft(m_vAq_ext,  'm_vAq_ext')
w_m_vAq_scrub = _ft(m_vAq_scrub,'m_vAq_scrub')
w_m_vAq_strip = _ft(m_vAq_strip,'m_vAq_strip')

grp5 = widgets.VBox([
    _header('Perturbation Window & Multipliers'),
    widgets.HBox([w_t_ini, w_t_fin]),
    widgets.HTML('<i style="font-size:11px">Feed multipliers (1.0 = no change)</i>'),
    widgets.HBox([w_m_Nd_aq, w_m_Pr_aq, w_m_H_aq]),
    widgets.HBox([w_m_Nd_scrub, w_m_H_scrub, w_m_H_strip]),
    widgets.HTML('<i style="font-size:11px">Flow multipliers</i>'),
    widgets.HBox([w_m_vO, w_m_vAq_ext, w_m_vAq_scrub, w_m_vAq_strip]),
])

# ── RUN button ────────────────────────────────────────────────────────────────
btn_run = widgets.Button(
    description='▶  RUN MODEL',
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px'),
    style={'font_weight': 'bold'},
)
out = widgets.Output()

ui = widgets.VBox([
    widgets.HTML('<hr style="border:2px solid #1F497D"/>'),
    grp1,
    widgets.HTML('<hr/>'),
    grp2,
    widgets.HTML('<hr/>'),
    grp3,
    widgets.HTML('<hr/>'),
    grp4,
    widgets.HTML('<hr/>'),
    grp5,
    widgets.HTML('<hr style="border:2px solid #1F497D"/>'),
    btn_run,
    out,
])

display(ui)

## Model Engine

The solver below implements the dynamic relaxed-iteration scheme. Each iteration corresponds to one time step $\Delta t$. Stages are updated in a single forward sweep per iteration using the separate-target relaxation pattern.

In [ ]:
def run_sx_model(p: Dict[str, Any]) -> Dict[str, Any]:
    """
    Runs the dynamic Nd/Pr solvent extraction model.

    Parameters
    ----------
    p : dict
        All model parameters (see Cell 6).

    Returns
    -------
    dict with keys:
        history       : ndarray (n_iter, n_stages, 8)  — C_Nd, C_Pr, Cbar_Nd,
                        Cbar_Pr, H, D_Nd, D_Pr, FreeLig for each stage/step
        time          : ndarray (n_iter,)  — time axis in hours
        residuals     : ndarray (n_iter,)  — max |delta| per iteration
        mass_balance  : dict
        convergence   : dict
    """
    # ── Unpack parameters ────────────────────────────────────────────────────
    a_nd  = p['a_Nd'];      b_nd  = p['b_Nd']
    a_pr  = p['a_Pr'];      b_pr  = p['b_Pr']
    L_tot = p['LIGAND_TOTAL']
    mw_nd = p['MW_Nd'];     mw_pr = p['MW_Pr']
    L_fl  = p['LIG_FLOOR']; H_fl  = p['H_FLOOR']

    vO_nom        = p['vO']
    vAq_ext_nom   = p['vAq_ext']
    vAq_scrub_nom = p['vAq_scrub']
    vAq_strip_nom = p['vAq_strip']

    Nd_aq  = p['Nd_aq_feed'];  Pr_aq  = p['Pr_aq_feed'];  H_aq  = p['H_aq_feed']
    Nd_sc  = p['Nd_scrub_feed'];                           H_sc  = p['H_scrub_feed']
    H_st   = p['H_strip_feed']

    alpha  = p['relax']
    n_iter = p['n_iter']
    dt     = p['dt']
    H_ini  = p['H_init']
    tol    = p['tol']

    t_ini = p['t_ini']; t_fin = p['t_fin']
    m_Nd_aq     = p['m_Nd_aq'];     m_Pr_aq     = p['m_Pr_aq']
    m_H_aq      = p['m_H_aq'];      m_Nd_scrub  = p['m_Nd_scrub']
    m_H_scrub   = p['m_H_scrub'];   m_H_strip   = p['m_H_strip']
    m_vO        = p['m_vO']
    m_vAq_ext   = p['m_vAq_ext'];   m_vAq_scrub = p['m_vAq_scrub']
    m_vAq_strip = p['m_vAq_strip']

    N = 10  # total stages (0-indexed: 0–9)

    # ── State arrays ─────────────────────────────────────────────────────────
    C_Nd    = np.zeros(N)        # aqueous Nd  (g/L)
    C_Pr    = np.zeros(N)        # aqueous Pr  (g/L)
    Cbar_Nd = np.zeros(N)        # organic Nd  (g/L)
    Cbar_Pr = np.zeros(N)        # organic Pr  (g/L)
    H       = np.full(N, H_ini)  # [H+]        (mol/L)

    # vars stored: C_Nd, C_Pr, Cbar_Nd, Cbar_Pr, H, D_Nd, D_Pr, FreeLig
    history   = np.zeros((n_iter, N, 8))
    residuals = np.zeros(n_iter)
    time_axis = np.arange(1, n_iter + 1) * dt

    # ── Main iteration loop ───────────────────────────────────────────────────
    for it in range(n_iter):
        t = (it + 1) * dt
        active = 1.0 if (t_ini <= t <= t_fin) else 0.0

        def _m(nominal, mult):
            return nominal * (1.0 + (mult - 1.0) * active)

        vO_eff        = _m(vO_nom,        m_vO)
        vAq_ext_eff   = _m(vAq_ext_nom,   m_vAq_ext)
        vAq_scrub_eff = _m(vAq_scrub_nom, m_vAq_scrub)
        vAq_strip_eff = _m(vAq_strip_nom, m_vAq_strip)

        Nd_aq_eff = _m(Nd_aq, m_Nd_aq);  Pr_aq_eff = _m(Pr_aq, m_Pr_aq)
        H_aq_eff  = _m(H_aq,  m_H_aq)
        Nd_sc_eff = _m(Nd_sc, m_Nd_scrub)
        H_sc_eff  = _m(H_sc,  m_H_scrub)
        H_st_eff  = _m(H_st,  m_H_strip)

        def _vAq_eff(s: int) -> float:
            if s <= 1:  return vAq_ext_eff
            if s <= 5:  return vAq_scrub_eff
            return vAq_strip_eff

        C_Nd_old    = C_Nd.copy()
        C_Pr_old    = C_Pr.copy()
        Cbar_Nd_old = Cbar_Nd.copy()
        Cbar_Pr_old = Cbar_Pr.copy()
        H_old       = H.copy()

        C_Nd_new    = C_Nd.copy()
        C_Pr_new    = C_Pr.copy()
        Cbar_Nd_new = Cbar_Nd.copy()
        Cbar_Pr_new = Cbar_Pr.copy()
        H_new       = H.copy()

        D_Nd_arr = np.zeros(N)
        D_Pr_arr = np.zeros(N)
        FL_arr   = np.zeros(N)

        for s in range(N):
            # Organic inlet: countercurrent, stage s receives organic from stage s-1
            # Stage 0 receives recycle from stage 9
            prev_org   = (s - 1) % N
            Cbar_in_Nd = Cbar_Nd_old[prev_org]
            Cbar_in_Pr = Cbar_Pr_old[prev_org]

            # Aqueous inlet (countercurrent to organic within each section)
            # Extraction (0-1): aq enters at index 1, exits at index 0
            # Scrub     (2-5): aq enters at index 5, exits at index 2
            # Strip     (6-9): aq enters at index 9, exits at index 6
            if s == 0:
                C_in_Nd = C_Nd_old[1];  C_in_Pr = C_Pr_old[1];  H_in = H_old[1]
            elif s == 1:
                C_in_Nd = Nd_aq_eff;    C_in_Pr = Pr_aq_eff;     H_in = H_aq_eff
            elif 2 <= s <= 4:
                C_in_Nd = C_Nd_old[s+1]; C_in_Pr = C_Pr_old[s+1]; H_in = H_old[s+1]
            elif s == 5:
                C_in_Nd = Nd_sc_eff;    C_in_Pr = 0.0;            H_in = H_sc_eff
            elif 6 <= s <= 8:
                C_in_Nd = C_Nd_old[s+1]; C_in_Pr = C_Pr_old[s+1]; H_in = H_old[s+1]
            else:  # s == 9
                C_in_Nd = 0.0;           C_in_Pr = 0.0;            H_in = H_st_eff

            vAq    = _vAq_eff(s)
            H_safe = max(H_old[s], H_fl)

            free_lig = max(
                L_tot - 3.0 * (Cbar_Nd_old[s] / mw_nd + Cbar_Pr_old[s] / mw_pr),
                L_fl
            )
            FL_arr[s] = free_lig

            # Distribution coefficients from PREVIOUS-ROW state
            D_Nd = a_nd * (H_safe ** b_nd) * (free_lig ** 3) / (H_safe ** 3)
            D_Pr = a_pr * (H_safe ** b_pr) * (free_lig ** 3) / (H_safe ** 3)
            D_Nd_arr[s] = D_Nd
            D_Pr_arr[s] = D_Pr

            # Raw (unrelaxed) stage-balance targets
            Css_Nd      = (vO_eff * Cbar_in_Nd + vAq * C_in_Nd) / (vO_eff * D_Nd + vAq)
            Css_Pr      = (vO_eff * Cbar_in_Pr + vAq * C_in_Pr) / (vO_eff * D_Pr + vAq)
            Cbar_tgt_Nd = D_Nd * Css_Nd
            Cbar_tgt_Pr = D_Pr * Css_Pr

            acid_rel = 3.0 * vO_eff * (
                (Cbar_tgt_Nd - Cbar_in_Nd) / mw_nd +
                (Cbar_tgt_Pr - Cbar_in_Pr) / mw_pr
            )
            Hss = max(H_in + acid_rel / vAq, H_fl)

            # CORRECT separate-target relaxation (no chaining)
            C_Nd_new[s]    = C_Nd_old[s]    + alpha * (Css_Nd      - C_Nd_old[s])
            C_Pr_new[s]    = C_Pr_old[s]    + alpha * (Css_Pr      - C_Pr_old[s])
            Cbar_Nd_new[s] = Cbar_Nd_old[s] + alpha * (Cbar_tgt_Nd - Cbar_Nd_old[s])
            Cbar_Pr_new[s] = Cbar_Pr_old[s] + alpha * (Cbar_tgt_Pr - Cbar_Pr_old[s])
            H_new[s]       = max(H_old[s] + alpha * (Hss - H_old[s]), H_fl)

        C_Nd    = C_Nd_new
        C_Pr    = C_Pr_new
        Cbar_Nd = Cbar_Nd_new
        Cbar_Pr = Cbar_Pr_new
        H       = H_new

        res = max(
            np.max(np.abs(C_Nd    - C_Nd_old)),
            np.max(np.abs(C_Pr    - C_Pr_old)),
            np.max(np.abs(Cbar_Nd - Cbar_Nd_old)),
            np.max(np.abs(Cbar_Pr - Cbar_Pr_old)),
        )
        residuals[it] = res

        history[it, :, 0] = C_Nd
        history[it, :, 1] = C_Pr
        history[it, :, 2] = Cbar_Nd
        history[it, :, 3] = Cbar_Pr
        history[it, :, 4] = H
        history[it, :, 5] = D_Nd_arr
        history[it, :, 6] = D_Pr_arr
        history[it, :, 7] = FL_arr

        if (it + 1) % 50 == 0:
            print(f"  Iter {it+1:4d} / {n_iter} | t = {t:6.1f} h | "
                  f"max|ΔC| = {res:.3e} | "
                  f"S1 Nd={C_Nd[0]:.3f} Pr={C_Pr[0]:.3f} g/L | "
                  f"S7 Nd={C_Nd[6]:.3f} Pr={C_Pr[6]:.3f} g/L")

    converged = residuals[-1] < tol
    print(f"\nDone. Final max|ΔC| = {residuals[-1]:.3e} "
          f"({'CONVERGED' if converged else 'NOT CONVERGED'}, tol={tol:.1e})")

    # ── Mass balance (uses final SS values; scrub exit at index 2 included) ──
    # Inputs: aq feed (index 1) + scrub feed (index 5)
    Nd_in_total  = Nd_aq * vAq_ext_nom + Nd_sc * vAq_scrub_nom  # g/min
    Pr_in_total  = Pr_aq * vAq_ext_nom
    # Outputs: raffinate (index 0) + scrub exit (index 2) + strip product (index 6)
    Nd_out_total = (C_Nd[0] * vAq_ext_nom
                    + C_Nd[2] * vAq_scrub_nom
                    + C_Nd[6] * vAq_strip_nom)
    Pr_out_total = (C_Pr[0] * vAq_ext_nom
                    + C_Pr[2] * vAq_scrub_nom
                    + C_Pr[6] * vAq_strip_nom)
    mb = {
        'Nd_in_total':  Nd_in_total,
        'Nd_out_total': Nd_out_total,
        'Pr_in_total':  Pr_in_total,
        'Pr_out_total': Pr_out_total,
        'Nd_error':     abs(Nd_in_total - Nd_out_total),
        'Pr_error':     abs(Pr_in_total - Pr_out_total),
    }

    settle_delta = float(np.max(np.abs(history[-1, :, :5] - history[-2, :, :5])))

    return {
        'history':       history,
        'time':          time_axis,
        'residuals':     residuals,
        'final_C_Nd':    C_Nd,
        'final_C_Pr':    C_Pr,
        'final_Cbar_Nd': Cbar_Nd,
        'final_Cbar_Pr': Cbar_Pr,
        'final_H':       H,
        'mass_balance':  mb,
        'convergence':   {'converged': converged, 'final_res': residuals[-1],
                          'settle_delta': settle_delta},
        'params':        p,
    }


In [ ]:
def display_results(result: Dict[str, Any]) -> None:
    """
    Creates and displays all diagnostic plots and tables for a model run.
    """
    history   = result['history']
    time      = result['time']
    residuals = result['residuals']
    p         = result['params']
    t_ini     = p['t_ini']
    t_fin     = p['t_fin']
    tol       = p['tol']

    # ── Colour palette ────────────────────────────────────────────────────────
    COL_ND  = '#2E75B6'
    COL_PR  = '#ED7D31'
    COL_S1  = '#70AD47'
    COL_S2  = '#4472C4'
    COL_S7  = '#C00000'

    _layout = dict(
        width=700, height=400,
        plot_bgcolor='white',
        paper_bgcolor='white',
        xaxis=dict(showgrid=True, gridcolor='#E0E0E0', title='Time (h)'),
        yaxis=dict(showgrid=True, gridcolor='#E0E0E0'),
        legend=dict(bgcolor='rgba(0,0,0,0)'),
    )

    def _pert_vrect(fig):
        fig.add_vrect(
            x0=t_ini, x1=t_fin,
            fillcolor='rgba(255,200,0,0.15)', line_width=0,
            annotation_text='Perturbation', annotation_position='top left',
            annotation_font_size=10,
        )
        fig.add_vline(x=t_ini, line_dash='dash', line_color='gray', line_width=1)
        fig.add_vline(x=t_fin, line_dash='dash', line_color='gray', line_width=1)

    # ── Plot 1 — Stage 1 aqueous (raffinate end) ──────────────────────────────
    fig1 = go.Figure()
    fig1.add_trace(go.Scatter(x=time, y=history[:, 0, 0], name='Nd', line=dict(color=COL_ND, width=2)))
    fig1.add_trace(go.Scatter(x=time, y=history[:, 0, 1], name='Pr', line=dict(color=COL_PR, width=2)))
    _pert_vrect(fig1)
    fig1.update_layout(**_layout,
        title='Stage 1 (Raffinate) — Aqueous Nd & Pr vs. Time',
        yaxis_title='Concentration (g/L)')
    fig1.show()

    # ── Plot 2 — Stage 7 aqueous (strip product) ──────────────────────────────
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(x=time, y=history[:, 6, 0], name='Nd', line=dict(color=COL_ND, width=2)))
    fig2.add_trace(go.Scatter(x=time, y=history[:, 6, 1], name='Pr', line=dict(color=COL_PR, width=2)))
    _pert_vrect(fig2)
    fig2.update_layout(**_layout,
        title='Stage 7 (Strip Product) — Aqueous Nd & Pr vs. Time',
        yaxis_title='Concentration (g/L)')
    fig2.show()

    # ── Plot 3 — [H+] at stages 1, 2, 7 ─────────────────────────────────────
    fig3 = go.Figure()
    fig3.add_trace(go.Scatter(x=time, y=history[:, 0, 4], name='Stage 1', line=dict(color=COL_S1, width=2)))
    fig3.add_trace(go.Scatter(x=time, y=history[:, 1, 4], name='Stage 2', line=dict(color=COL_S2, width=2)))
    fig3.add_trace(go.Scatter(x=time, y=history[:, 6, 4], name='Stage 7', line=dict(color=COL_S7, width=2)))
    _pert_vrect(fig3)
    fig3.update_layout(**_layout,
        title='[H⁺] vs. Time — Stages 1, 2 & 7',
        yaxis_title='[H⁺] (mol/L)')
    fig3.show()

    # ── Plot 4 — Loaded organic at stage 2 ───────────────────────────────────
    fig4 = go.Figure()
    fig4.add_trace(go.Scatter(x=time, y=history[:, 1, 2], name='Nd̄', line=dict(color=COL_ND, width=2)))
    fig4.add_trace(go.Scatter(x=time, y=history[:, 1, 3], name='P̄r', line=dict(color=COL_PR, width=2)))
    _pert_vrect(fig4)
    fig4.update_layout(**_layout,
        title='Loaded Organic at Stage 2 vs. Time',
        yaxis_title='Organic concentration (g/L)')
    fig4.show()

    # ── Plot 5 — Convergence ──────────────────────────────────────────────────
    fig5 = go.Figure()
    fig5.add_trace(go.Scatter(
        x=np.arange(1, len(residuals) + 1), y=residuals,
        name='max|ΔC|', line=dict(color='#1F497D', width=2)
    ))
    fig5.add_hline(y=tol, line_dash='dot', line_color='red',
                   annotation_text=f'tol = {tol:.0e}',
                   annotation_position='right')
    fig5.update_layout(**_layout,
        title='Convergence — Max |ΔC| per Iteration',
        xaxis_title='Iteration',
        yaxis_title='Max |ΔC| (g/L or mol/L)',
        yaxis_type='log')
    fig5.show()

    # ── Results Table 1 — Final stage profile ─────────────────────────────────
    sections = (['Extraction'] * 2 + ['Scrub'] * 4 + ['Strip'] * 4)
    C_nd_f    = result['final_C_Nd']
    C_pr_f    = result['final_C_Pr']
    Cb_nd_f   = result['final_Cbar_Nd']
    Cb_pr_f   = result['final_Cbar_Pr']
    H_f       = result['final_H']

    aq_total  = C_nd_f + C_pr_f + 1e-15
    org_total = Cb_nd_f + Cb_pr_f + 1e-15

    df = pd.DataFrame({
        'Stage':        np.arange(1, 11),
        'Section':      sections,
        'C_Nd (g/L)':   np.round(C_nd_f, 4),
        'C_Pr (g/L)':   np.round(C_pr_f, 4),
        'Cbar_Nd (g/L)':np.round(Cb_nd_f, 4),
        'Cbar_Pr (g/L)':np.round(Cb_pr_f, 4),
        '[H+] (mol/L)': np.round(H_f, 5),
        '%Nd aq':       np.round(100 * C_nd_f / aq_total, 2),
        '%Pr aq':       np.round(100 * C_pr_f / aq_total, 2),
        '%Nd org':      np.round(100 * Cb_nd_f / org_total, 2),
        '%Pr org':      np.round(100 * Cb_pr_f / org_total, 2),
    })

    def _style_df(df):
        styles = []
        for i in range(len(df)):
            bg = '#DEEAF1' if i % 2 == 0 else 'white'
            styles.append({'selector': f'tr:nth-child({i+2})', 'props': [('background-color', bg)]})
        return (df.style
                .set_table_styles([
                    {'selector': 'th', 'props': [
                        ('background-color', '#1F497D'), ('color', 'white'),
                        ('font-weight', 'bold'), ('text-align', 'center')]},
                    {'selector': 'td', 'props': [('text-align', 'center'), ('padding', '4px 10px')]},
                ] + styles)
                .hide(axis='index'))

    display(HTML('<h3>Final Stage Profile</h3>'))
    display(_style_df(df))

    # ── Results Table 2 — Validation ──────────────────────────────────────────
    mb  = result['mass_balance']
    cv  = result['convergence']

    # Strip product purity (stage 7, index 6)
    s7_nd = C_nd_f[6]; s7_pr = C_pr_f[6]
    s7_tot = s7_nd + s7_pr + 1e-15
    pct_nd = 100 * s7_nd / s7_tot
    pct_pr = 100 * s7_pr / s7_tot

    val_df = pd.DataFrame({
        'Metric': [
            'Strip product %Nd (this model)',
            'Strip product %Nd (Lyon model)',
            'Strip product %Nd (Lyon expt.)',
            'Strip product %Pr (this model)',
            'Strip product %Pr (Lyon model)',
            'Strip product %Pr (Lyon expt.)',
            'Nd mass balance error (g/min)',
            'Pr mass balance error (g/min)',
            'Settling check max|ΔC| (last step)',
            'Converged?',
        ],
        'Value': [
            f'{pct_nd:.2f}%', '97.2%', '96.7%',
            f'{pct_pr:.2f}%', '2.8%',  '3.3%',
            f'{mb["Nd_error"]:.2e}',
            f'{mb["Pr_error"]:.2e}',
            f'{cv["settle_delta"]:.2e}',
            '✓ YES' if cv['converged'] else '✗ NO',
        ],
    })
    display(HTML('<h3>Validation vs. Lyon et al. (2017)</h3>'))
    display(_style_df(val_df))

In [ ]:
def _collect_params() -> Dict[str, Any]:
    """Read all widget values into a params dict."""
    return {
        # Equilibrium
        'a_Nd': w_a_Nd.value,          'b_Nd': w_b_Nd.value,
        'a_Pr': w_a_Pr.value,          'b_Pr': w_b_Pr.value,
        'LIGAND_TOTAL': w_LIGAND_TOTAL.value,
        'LIG_FLOOR': LIG_FLOOR,        'MW_Nd': MW_Nd,  'MW_Pr': MW_Pr,
        'H_FLOOR': H_FLOOR,
        # Flows
        'vO': w_vO.value,
        'vAq_ext': w_vAq_ext.value,    'vAq_scrub': w_vAq_scrub.value,
        'vAq_strip': w_vAq_strip.value,
        # Feeds
        'Nd_aq_feed': w_Nd_aq.value,   'Pr_aq_feed': w_Pr_aq.value,
        'H_aq_feed': w_H_aq.value,
        'Nd_scrub_feed': w_Nd_scrub.value, 'H_scrub_feed': w_H_scrub.value,
        'H_strip_feed': w_H_strip.value,
        # Solver
        'relax': w_relax.value,         'n_iter': w_n_iter.value,
        'dt': w_dt.value,               'H_init': w_H_init.value,
        'tol': w_tol.value,
        # Perturbation
        't_ini': w_t_ini.value,         't_fin': w_t_fin.value,
        'm_Nd_aq': w_m_Nd_aq.value,     'm_Pr_aq': w_m_Pr_aq.value,
        'm_H_aq': w_m_H_aq.value,
        'm_Nd_scrub': w_m_Nd_scrub.value,'m_H_scrub': w_m_H_scrub.value,
        'm_H_strip': w_m_H_strip.value,
        'm_vO': w_m_vO.value,
        'm_vAq_ext': w_m_vAq_ext.value, 'm_vAq_scrub': w_m_vAq_scrub.value,
        'm_vAq_strip': w_m_vAq_strip.value,
    }


def _on_run(_button_event=None):
    with out:
        out.clear_output(wait=True)
        print('Running model...')
        params = _collect_params()
        result = run_sx_model(params)
        display_results(result)


btn_run.on_click(_on_run)

# Auto-run on notebook load so the user sees results immediately
_on_run()